# Chapter 6: Sequential Recommendation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kimfalk/modern-recommender-systems/blob/main/notebooks/chapter-06/sasrec.ipynb)



# SASRec on MovieLens 25M — companion notebook (Chapter 6)

This notebook implements the SASRec model exactly as it appears in **Chapter 6, Section 6.3**
of *Modern Recommender Systems*: the architecture from **Listing 6.1**, the gBCE loss from
**Listing 6.2**, and the three ablations from **Listing 6.3**. It trains on MovieLens 25M with
a per-user leave-one-out time split, matching the protocol described in Section 6.7.1.

**Before you run this:**
- Training the full model on MovieLens 25M is the setup described in the chapter
  ("trains in roughly 40 minutes on a single A100"). On a laptop CPU this will be much
  slower. Set `QUICK_TEST = True` below to subsample users and reduce epochs so you can
  verify everything runs end-to-end in a few minutes before committing to a full run.



In [ ]:
# Install datasets library for HuggingFace
!pip install -q datasets
!pip install -q mlflow  
!pip install -q torch pandas numpy requests

In [ ]:
from recsys.utils.colab import setup_colab_environment, get_data_path, check_gpu


In [ ]:
# One-line setup for Colab users
setup_colab_environment()

# Check GPU availability
check_gpu()

In [ ]:

QUICK_TEST = False   # set to False to reproduce the full Chapter 6 setup on MovieLens 25M

import os
import zipfile
import time
import math
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from typing import List, Optional

import matplotlib.pyplot as plt
from recsys.data import loaders

from recsys.data.loaders import (
    load_movielens,
    load_movielens_links,
    load_tmdb_movie_descriptions,
)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DATA_PATH = get_data_path()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



## 1. Data: downloading and preparing MovieLens 25M

Section 6.1.2 explains why MovieLens 25M is used despite not being a naturally sequential
dataset: each user has a chronological list of ratings, which we treat as implicit feedback
by discarding the rating value and keeping only the (user, item, timestamp) interaction —
matching the "Implicit vs. explicit feedback" callout in Section 6.2.


In [ ]:

# Cell 4: Load movie data
ratings, movies = load_movielens(
  dataset='ml-25m', data_dir=DATA_PATH
)

print(f"Ratings: {len(ratings):,}")
print(f"Movies: {len(movies):,}")


In [ ]:

ratings = ratings.sort_values(["userId", "timestamp"]).reset_index(drop=True)

if QUICK_TEST:
    # Subsample a manageable number of users so the rest of the notebook runs in minutes.
    sample_users = ratings["userId"].drop_duplicates().sample(n=30000, random_state=SEED)
    ratings = ratings[ratings["userId"].isin(sample_users)].reset_index(drop=True)

print(f"Interactions: {len(ratings):,}")
print(f"Users: {ratings['userId'].nunique():,}")
print(f"Items: {ratings['movieId'].nunique():,}")


In [ ]:

# Re-index items to a contiguous range starting at 1 (0 is reserved for padding, per Listing 6.1).
item_ids = ratings["movieId"].unique()
item_id_map = {old: new for new, old in enumerate(sorted(item_ids), start=1)}
ratings["item_idx"] = ratings["movieId"].map(item_id_map)
num_items = len(item_id_map)
print(f"num_items (re-indexed, 1..{num_items}): {num_items:,}")

# Build each user's chronological sequence of item indices.
user_sequences = (
    ratings.groupby("userId")["item_idx"]
    .apply(list)
    .reset_index(drop=True)
)
# Drop users with fewer than 3 interactions: we need at least one each for train/val/test.
user_sequences = user_sequences[user_sequences.apply(len) >= 3].reset_index(drop=True)
print(f"Usable users (>=3 interactions): {len(user_sequences):,}")



## 2. Per-user leave-one-out time split

Section 6.7.1 specifies per-user leave-one-out by time for the SASRec experiments in this
chapter: the most recent interaction is the test target, the second-most-recent is the
validation target, and everything before that is training context.


In [ ]:

MAX_LEN = 200  # matches Listing 6.1's default max_len

def build_split(sequences):
    train_seqs, val_targets, test_targets, val_inputs, test_inputs = [], [], [], [], []
    for seq in sequences:
        train_part = seq[:-2]
        val_target = seq[-2]
        test_target = seq[-1]
        train_seqs.append(train_part)
        val_inputs.append(seq[:-2])      # context for predicting the val target
        val_targets.append(val_target)
        test_inputs.append(seq[:-1])     # context for predicting the test target
        test_targets.append(test_target)
    return train_seqs, val_inputs, val_targets, test_inputs, test_targets

train_seqs, val_inputs, val_targets, test_inputs, test_targets = build_split(
    user_sequences.tolist()
)
print(f"Train users: {len(train_seqs):,}")
print(f"Example train sequence (truncated): {train_seqs[0][:10]}...")


In [ ]:

def truncate_and_pad(seq, max_len=MAX_LEN):
    '''Truncate to the most recent max_len items; left-pad shorter sequences with 0.'''
    seq = seq[-max_len:]
    pad_len = max_len - len(seq)
    return [0] * pad_len + seq

class SASRecTrainDataset(Dataset):
    '''Each training example is one user's (input, target) pair built by shifting
    their history left by one position, as described in Section 6.3.2's training loop.'''

    def __init__(self, sequences, num_items, max_len=MAX_LEN, n_neg=256):
        self.sequences = [s for s in sequences if len(s) >= 2]
        self.num_items = num_items
        self.max_len = max_len
        self.n_neg = n_neg

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        input_seq = truncate_and_pad(seq[:-1], self.max_len)
        target_seq = truncate_and_pad(seq[1:], self.max_len)
        input_seq = torch.tensor(input_seq, dtype=torch.long)
        target_seq = torch.tensor(target_seq, dtype=torch.long)
        # Sample negatives per position, avoiding item 0 (padding).
        neg_items = torch.randint(1, self.num_items + 1, (self.max_len, self.n_neg))
        return input_seq, target_seq, neg_items

class SASRecEvalDataset(Dataset):
    '''One example per user: the context sequence and the single held-out target.'''

    def __init__(self, input_seqs, targets, max_len=MAX_LEN):
        self.input_seqs = input_seqs
        self.targets = targets
        self.max_len = max_len

    def __len__(self):
        return len(self.input_seqs)

    def __getitem__(self, idx):
        seq = truncate_and_pad(self.input_seqs[idx], self.max_len)
        return torch.tensor(seq, dtype=torch.long), torch.tensor(self.targets[idx], dtype=torch.long)



## 3. The SASRec architecture (Listing 6.1)

Copied directly from the chapter: item embeddings, learned positional embeddings, a causal
mask, a padding mask, and a stack of pre-norm Transformer encoder layers via PyTorch's
built-in `nn.TransformerEncoder`.


In [ ]:

class SASRec(nn.Module):
    def __init__(self, num_items, max_len=200, hidden_dim=64,
                 num_layers=2, num_heads=2, dropout=0.2, use_pos_emb=True):
        super().__init__()
        self.num_items = num_items
        self.max_len = max_len
        self.use_pos_emb = use_pos_emb  # toggle for Ablation A, see Section 6.3.2
        self.item_emb = nn.Embedding(num_items + 1, hidden_dim, padding_idx=0)  # A
        self.pos_emb = nn.Embedding(max_len, hidden_dim)                       # B
        self.input_dropout = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=num_heads,
            dim_feedforward=hidden_dim, dropout=dropout,
            batch_first=True, norm_first=True,  # C
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.final_norm = nn.LayerNorm(hidden_dim)

    def forward(self, sequences):
        batch_size, seq_len = sequences.shape
        x = self.item_emb(sequences)
        if self.use_pos_emb:
            positions = torch.arange(seq_len, device=sequences.device)
            positions = positions.unsqueeze(0).expand(batch_size, -1)
            x = x + self.pos_emb(positions)
        x = self.input_dropout(x)
        causal_mask = torch.triu(  # D
            torch.ones(seq_len, seq_len, device=sequences.device, dtype=torch.bool),
            diagonal=1,
        )
        padding_mask = (sequences == 0)  # E
        x = self.transformer(x, mask=causal_mask, src_key_padding_mask=padding_mask)
        return self.final_norm(x)

# A Item embeddings, with index 0 reserved for the padding token.
# B Learned positional embeddings, one vector per position in the sequence.
# C Pre-norm Transformer block: LayerNorm runs before attention, which improves training stability.
# D The causal mask blocks position i from attending to positions j > i.
# E The padding mask tells attention to ignore padded positions on the left.



## 4. Training with gBCE (Listing 6.2)

The generalized binary cross-entropy loss from Petrov & Macdonald's gSASRec paper (2023),
which corrects the overconfidence problem that plain BCE with sampled negatives produces.


In [ ]:

def gbce_loss(pos_scores, neg_scores, num_items, t=0.75):
    '''
    pos_scores: (batch, seq_len)         scores for the true next item at each position
    neg_scores: (batch, seq_len, n_neg)  scores for sampled negatives at each position
    num_items:  total catalogue size, used to compute the sampling rate
    t:          gBCE calibration parameter; t=0 recovers vanilla BCE
    '''
    n_neg = neg_scores.shape[-1]
    alpha = n_neg / max(num_items - 1, 1)                                        # A
    beta = alpha * (t * (1.0 - 1.0 / alpha) + 1.0 / alpha)                       # B
    # Positive contribution: -beta * log sigmoid(s+) = beta * softplus(-s+)
    pos_loss = beta * F.softplus(-pos_scores)                                    # C
    # Negative contribution: -log(1 - sigmoid(s-)) = softplus(s-), summed across negatives
    neg_loss = F.softplus(neg_scores).sum(dim=-1)                                # D
    return (pos_loss + neg_loss).mean() / (n_neg + 1)

# A Sampling rate: the fraction of the catalogue that appears as a negative for each positive.
# B The calibration exponent; t=0 makes beta=1 (vanilla BCE), t=1 maximally corrects.
# C Numerically stable form of -beta * log sigmoid(score).
# D Numerically stable form of -log(1 - sigmoid(score)).


In [ ]:

def train_sasrec(model, train_dataset, num_epochs, batch_size=128, lr=1e-3, t=0.75):
    model.to(device)
    loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(num_epochs):
        model.train()
        total_loss, n_batches = 0.0, 0
        t0 = time.time()
        for input_seq, target_seq, neg_items in loader:
            input_seq = input_seq.to(device)
            target_seq = target_seq.to(device)
            neg_items = neg_items.to(device)

            hidden = model(input_seq)                          # (batch, seq_len, hidden_dim)

            pos_mask = (target_seq != 0)
            pos_item_emb = model.item_emb(target_seq)           # (batch, seq_len, hidden_dim)
            pos_scores = (hidden * pos_item_emb).sum(-1)         # (batch, seq_len)

            neg_item_emb = model.item_emb(neg_items)             # (batch, seq_len, n_neg, hidden_dim)
            neg_scores = (hidden.unsqueeze(2) * neg_item_emb).sum(-1)  # (batch, seq_len, n_neg)

            # Zero out the loss contribution from padded positions.
            pos_scores = pos_scores.masked_fill(~pos_mask, 0.0)
            neg_scores = neg_scores.masked_fill(~pos_mask.unsqueeze(-1), 0.0)

            loss = gbce_loss(pos_scores, neg_scores, model.num_items, t=t)
            opt.zero_grad()
            loss.backward()
            opt.step()

            total_loss += loss.item()
            n_batches += 1

        elapsed = time.time() - t0
        print(f"Epoch {epoch+1}/{num_epochs}: loss={total_loss/n_batches:.4f}  ({elapsed:.1f}s)")
    return model


In [ ]:

N_EPOCHS = 2 if QUICK_TEST else 20
HIDDEN_DIM = 64
N_NEG = 256 if not QUICK_TEST else 64

train_dataset = SASRecTrainDataset(train_seqs, num_items, max_len=MAX_LEN, n_neg=N_NEG)

model = SASRec(num_items=num_items, max_len=MAX_LEN, hidden_dim=HIDDEN_DIM,
                num_layers=2, num_heads=2, dropout=0.2, use_pos_emb=True)

model = train_sasrec(model, train_dataset, num_epochs=N_EPOCHS, batch_size=128)



## 5. Evaluation: NDCG@10 and HR@10 on the held-out test target

Full-catalogue evaluation, as discussed in Section 6.3.2 — scores against every item in
`model.item_emb.weight`, not a sampled subset.


In [ ]:

def evaluate(model, eval_dataset, k=10, batch_size=256):
    model.eval()
    loader = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False)
    ndcgs, hits = [], []
    with torch.no_grad():
        for seqs, targets in loader:
            seqs = seqs.to(device)
            targets = targets.to(device)
            hidden = model(seqs)
            user_state = hidden[:, -1, :]
            scores = user_state @ model.item_emb.weight.T  # (batch, num_items+1)
            scores[:, 0] = -float("inf")  # never recommend the padding token

            topk = torch.topk(scores, k, dim=1).indices
            hit = (topk == targets.unsqueeze(1))
            rank = hit.float().argmax(dim=1)
            has_hit = hit.any(dim=1)
            ndcg = torch.where(has_hit, 1.0 / torch.log2(rank.float() + 2),
                                torch.zeros_like(rank, dtype=torch.float))
            ndcgs.extend(ndcg.cpu().tolist())
            hits.extend(has_hit.float().cpu().tolist())
    return {"NDCG@10": float(np.mean(ndcgs)), "HR@10": float(np.mean(hits))}

test_dataset = SASRecEvalDataset(test_inputs, test_targets, max_len=MAX_LEN)
test_metrics = evaluate(model, test_dataset)
print(f"Test set — NDCG@10: {test_metrics['NDCG@10']:.4f}   HR@10: {test_metrics['HR@10']:.4f}")



## 6. Three ablations (Listing 6.3)

Isolating what the architecture is actually doing, as described in Section 6.3.2:

- **Ablation A — remove positional embeddings.** Requires retraining, since this changes
  what the model learns. We train a second model with `use_pos_emb=False`.
- **Ablation B — shuffle the history at inference time.** Uses the already-trained model
  from Section 5 above; no retraining needed.
- **Ablation C — keep only the most recent item.** Also uses the already-trained model.


In [ ]:

def evaluate_ablation(model, eval_dataset, mode='full', batch_size=256, seed=SEED):
    '''
    mode:
      'full'       - standard evaluation, history fed as-is
      'shuffled'   - history is randomly permuted before scoring
      'last_only'  - history is truncated to the single most recent item
    '''
    model.eval()
    rng = torch.Generator().manual_seed(seed)
    loader = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False)
    ndcgs, hits = [], []
    with torch.no_grad():
        for sequences, targets in loader:
            sequences = sequences.clone()
            if mode == 'shuffled':
                for i in range(sequences.shape[0]):
                    valid = sequences[i] != 0
                    n_valid = int(valid.sum())
                    if n_valid > 1:
                        perm = torch.randperm(n_valid, generator=rng)
                        sequences[i, valid] = sequences[i, valid][perm]
            elif mode == 'last_only':
                last_item = sequences.gather(1, (sequences != 0).sum(1, keepdim=True) - 1)
                sequences = torch.zeros_like(sequences)
                sequences[:, -1] = last_item.squeeze(1)

            sequences = sequences.to(device)
            targets = targets.to(device)
            hidden = model(sequences)
            user_state = hidden[:, -1, :]                                            # A
            scores = user_state @ model.item_emb.weight.T                            # B
            scores[:, 0] = -float("inf")

            topk = torch.topk(scores, 10, dim=1).indices
            hit = (topk == targets.unsqueeze(1))
            rank = hit.float().argmax(dim=1)
            has_hit = hit.any(dim=1)
            ndcg = torch.where(has_hit, 1.0 / torch.log2(rank.float() + 2),
                                torch.zeros_like(rank, dtype=torch.float))
            ndcgs.extend(ndcg.cpu().tolist())
            hits.extend(has_hit.float().cpu().tolist())
    return {"NDCG@10": float(np.mean(ndcgs)), "HR@10": float(np.mean(hits))}

# A The user state vector is the hidden state at the rightmost position.
# B Scoring against every item is just a matrix multiplication against the embedding table.


In [ ]:

# Ablation A: a second model trained without positional embeddings.
N_EPOCHS_ABLATION_A = N_EPOCHS  # keep epochs comparable to the main model

model_no_pos = SASRec(num_items=num_items, max_len=MAX_LEN, hidden_dim=HIDDEN_DIM,
                       num_layers=2, num_heads=2, dropout=0.2, use_pos_emb=False)
model_no_pos = train_sasrec(model_no_pos, train_dataset, num_epochs=N_EPOCHS_ABLATION_A, batch_size=128)


In [ ]:

results = {}
results["Full model (with position)"] = evaluate_ablation(model, test_dataset, mode='full')
results["Ablation A: no positional embeddings"] = evaluate_ablation(model_no_pos, test_dataset, mode='full')
results["Ablation B: shuffled history at inference"] = evaluate_ablation(model, test_dataset, mode='shuffled')
results["Ablation C: last item only"] = evaluate_ablation(model, test_dataset, mode='last_only')

print(f"{'Setup':<42}{'NDCG@10':>10}{'HR@10':>10}")
print("-" * 62)
for name, metrics in results.items():
    print(f"{name:<42}{metrics['NDCG@10']:>10.4f}{metrics['HR@10']:>10.4f}")



## 7. Reading the results

Compare the four rows above the way Section 6.3.2 does:

- If **Ablation A** drops sharply relative to the full model, position is doing real work
  for this dataset at training time.
- If **Ablation B** (shuffled at inference) is close to the full model, the trained model
  isn't leaning heavily on exact order — consistent with the chapter's "more of a set
  model that mildly recency-weights" reading.
- If **Ablation C** (last item only) recovers most of the full model's NDCG@10, a much
  simpler Markov-1 baseline may be competitive for this dataset.

As discussed when we revisited this section, these particular numbers are a property of
*MovieLens 25M*, not a universal property of SASRec — Klenitskiy et al. (2024) found this
same shuffle test produces dramatically different results across datasets (near-total
collapse on 30Music, almost no effect on RetailRocket). Treat the table above as a
measurement of this dataset, and rerun it on your own data before drawing conclusions for
a different domain.

**To reproduce the full chapter setup:** set `QUICK_TEST = False` in the first cell and
rerun. On MovieLens 25M with `n_neg=256` and 20 epochs, expect roughly the training time
quoted in Section 6.3.2 on a single A100; CPU-only training will take substantially longer.
